# Hopfield 统一任务基准：Phase 1

状态：`self-contained / not-run`

这个 notebook 是单文件实验：不克隆仓库，不导入旁边的 Python 文件，也不安装额外依赖。它先用 Classical Hopfield 与三次 Polynomial DAM 验证统一任务框架。当前文件没有保存任何运行输出，因此不能据此得出模型优劣结论。

## 0. 组件式实验管线

`a 记忆输入 → b 模型存储 → c 检索线索 → d 检索动力学 → e 测量 → f 同图比较`

所有模型共用 `a/c/e/f`，只替换 `b/d`。运行器先生成一次模式和线索，再把同一个 tensor 交给所有模型；作图前再次检查每个 trial 是否包含完整模型集合。

In [ ]:
from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone
from time import perf_counter
from typing import Any, Callable, Iterable
from urllib.request import Request, urlopen
from uuid import uuid4
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.max_columns", 50)

SOURCE_NOTEBOOK = "hopfield-benchmark/hopfield_benchmark_phase1_colab.ipynb"
try:
    request = Request(
        "https://api.github.com/repos/Heptazero/nn-labs/commits/main",
        headers={"User-Agent": "hopfield-benchmark-colab"},
    )
    with urlopen(request, timeout=20) as response:
        SOURCE_COMMIT = json.load(response)["sha"]
except Exception as error:
    SOURCE_COMMIT = "unresolved"
    print(f"warning: could not resolve GitHub commit: {error}")

print(f"torch={torch.__version__}, pandas={pd.__version__}")
print(f"source commit={SOURCE_COMMIT}")

## 1. a / c：共享记忆与共享线索

`a1` 生成独立随机 `{-1,+1}` 模式。`c1` 精确翻转 `round(rho*N)` 个位置。线索在模型循环外生成，禁止每个模型各自抽一次噪声。

In [ ]:
@dataclass(frozen=True)
class MemorySet:
    patterns: torch.Tensor
    pattern_ids: tuple[int, ...]
    dataset_id: str
    data_seed: int
    encoding: str = "binary_pm1"

    @property
    def P(self) -> int:
        return int(self.patterns.shape[0])

    @property
    def N(self) -> int:
        return int(self.patterns.shape[1])


def make_generator(seed: int, device: torch.device | str = "cpu") -> torch.Generator:
    generator = torch.Generator(device=device)
    generator.manual_seed(int(seed))
    return generator


def a1_make_independent_binary(
    N: int,
    P: int,
    data_seed: int,
    *,
    device: torch.device | str = "cpu",
) -> MemorySet:
    if N <= 0 or P <= 0:
        raise ValueError("N and P must be positive")
    bits = torch.randint(
        0,
        2,
        (P, N),
        generator=make_generator(data_seed, device),
        device=device,
        dtype=torch.int8,
    )
    return MemorySet(
        patterns=bits.mul(2).sub(1),
        pattern_ids=tuple(range(P)),
        dataset_id="independent_binary",
        data_seed=int(data_seed),
    )


def c1_make_hamming_cue(
    target: torch.Tensor,
    corruption_level: float,
    cue_seed: int,
) -> torch.Tensor:
    if target.ndim != 1:
        raise ValueError("target must be one-dimensional")
    if not 0.0 <= corruption_level <= 1.0:
        raise ValueError("corruption_level must lie in [0, 1]")
    cue = target.detach().clone().to(torch.int8)
    flip_count = int(round(corruption_level * cue.numel()))
    if flip_count:
        indices = torch.randperm(
            cue.numel(),
            generator=make_generator(cue_seed, cue.device),
            device=cue.device,
        )[:flip_count]
        cue[indices] *= -1
    return cue

## 2. b / d：可替换模型组件

两种模型都实现 `fit → retrieve → resource_summary`。Classical Hopfield 使用零对角 Hebb 权重；Polynomial DAM 直接保存记忆表，并逐坐标比较 `+1/-1` 两个候选状态对应的三次 power energy。两者都采用随机顺序异步更新，零差值保持原状态。

In [ ]:
@dataclass
class RetrievalResult:
    final_state: torch.Tensor
    status: str
    sweeps: int
    state_updates: int
    retrieval_flops: int
    energy_trace: list[float] = field(default_factory=list)
    error_trace: list[float] = field(default_factory=list)
    diagnostics: dict[str, Any] = field(default_factory=dict)


def binary_state(state: torch.Tensor) -> torch.Tensor:
    values = set(torch.unique(state.to(torch.int8)).tolist())
    if state.ndim != 1 or not values.issubset({-1, 1}):
        raise ValueError("state must be one-dimensional and use {-1, +1}")
    return state.detach().clone().to(torch.float64)


class ClassicalHopfield:
    model_id = "classical_hebb"

    def fit(self, patterns: torch.Tensor) -> "ClassicalHopfield":
        binary = patterns.to(torch.float64)
        if binary.ndim != 2:
            raise ValueError("patterns must have shape [P, N]")
        self.N = int(binary.shape[1])
        self.weights = binary.T @ binary / self.N
        self.weights.fill_diagonal_(0.0)
        return self

    def energy(self, state: torch.Tensor) -> float:
        return float((-0.5 * state @ self.weights @ state).item())

    def retrieve(
        self,
        cue: torch.Tensor,
        *,
        target: torch.Tensor,
        update_seed: int,
        max_sweeps: int,
    ) -> RetrievalResult:
        if max_sweeps <= 0:
            raise ValueError("max_sweeps must be positive")
        state = binary_state(cue)
        target_f = target.to(torch.float64)
        generator = make_generator(update_seed, state.device)
        energies = [self.energy(state)]
        errors = [float(torch.mean((state != target_f).to(torch.float64)).item())]
        updates = 0
        first_sweep_unchanged = False
        for sweep in range(1, max_sweeps + 1):
            changed = False
            order = torch.randperm(self.N, generator=generator, device=state.device)
            for index in order.tolist():
                local_field = float(torch.dot(self.weights[index], state).item())
                new_value = (
                    1.0
                    if local_field > 0.0
                    else -1.0
                    if local_field < 0.0
                    else state[index].item()
                )
                if new_value != state[index].item():
                    state[index] = new_value
                    changed = True
                updates += 1
            energies.append(self.energy(state))
            errors.append(float(torch.mean((state != target_f).to(torch.float64)).item()))
            if sweep == 1:
                first_sweep_unchanged = not changed
            if not changed:
                return RetrievalResult(
                    state.to(torch.int8),
                    "fixed",
                    sweep,
                    updates,
                    updates * (2 * self.N - 1),
                    energies,
                    errors,
                    {"first_sweep_unchanged": first_sweep_unchanged},
                )
        return RetrievalResult(
            state.to(torch.int8),
            "max_steps",
            max_sweeps,
            updates,
            updates * (2 * self.N - 1),
            energies,
            errors,
            {"first_sweep_unchanged": first_sweep_unchanged},
        )

    def resource_summary(self) -> dict[str, int | str]:
        return {
            "model_config": "hebbian_zero_diagonal_async",
            "parameter_count": self.N * (self.N - 1) // 2,
            "storage_bytes": self.weights.numel() * self.weights.element_size(),
        }


class PolynomialDAM:
    def __init__(self, degree: int = 3) -> None:
        if degree < 2:
            raise ValueError("degree must be at least 2")
        self.degree = int(degree)
        self.model_id = f"polynomial_dam_d{self.degree}"

    def fit(self, patterns: torch.Tensor) -> "PolynomialDAM":
        binary = patterns.to(torch.float64)
        if binary.ndim != 2:
            raise ValueError("patterns must have shape [P, N]")
        self.patterns = binary.detach().clone()
        self.P, self.N = map(int, binary.shape)
        return self

    def energy(self, overlaps: torch.Tensor) -> float:
        return float(-torch.sum(overlaps.pow(self.degree)).item())

    def retrieve(
        self,
        cue: torch.Tensor,
        *,
        target: torch.Tensor,
        update_seed: int,
        max_sweeps: int,
    ) -> RetrievalResult:
        if max_sweeps <= 0:
            raise ValueError("max_sweeps must be positive")
        state = binary_state(cue)
        target_f = target.to(torch.float64)
        overlaps = self.patterns @ state / self.N
        generator = make_generator(update_seed, state.device)
        energies = [self.energy(overlaps)]
        errors = [float(torch.mean((state != target_f).to(torch.float64)).item())]
        updates = 0
        first_sweep_unchanged = False
        for sweep in range(1, max_sweeps + 1):
            changed = False
            order = torch.randperm(self.N, generator=generator, device=state.device)
            for index in order.tolist():
                coordinate = self.patterns[:, index] / self.N
                base = overlaps - coordinate * state[index]
                score_plus = torch.sum((base + coordinate).pow(self.degree))
                score_minus = torch.sum((base - coordinate).pow(self.degree))
                old_value = state[index].item()
                if score_plus > score_minus:
                    new_value = 1.0
                elif score_minus > score_plus:
                    new_value = -1.0
                else:
                    new_value = old_value
                if new_value != old_value:
                    state[index] = new_value
                    overlaps += coordinate * (new_value - old_value)
                    changed = True
                updates += 1
            energies.append(self.energy(overlaps))
            errors.append(float(torch.mean((state != target_f).to(torch.float64)).item()))
            if sweep == 1:
                first_sweep_unchanged = not changed
            diagnostics = {
                "degree": self.degree,
                "first_sweep_unchanged": first_sweep_unchanged,
            }
            if not changed:
                return RetrievalResult(
                    state.to(torch.int8),
                    "fixed",
                    sweep,
                    updates,
                    updates * (8 * self.P + 2),
                    energies,
                    errors,
                    diagnostics,
                )
        return RetrievalResult(
            state.to(torch.int8),
            "max_steps",
            max_sweeps,
            updates,
            updates * (8 * self.P + 2),
            energies,
            errors,
            diagnostics,
        )

    def resource_summary(self) -> dict[str, int | str]:
        return {
            "model_config": f"power_energy_degree_{self.degree}_async",
            "parameter_count": self.P * self.N,
            "storage_bytes": self.patterns.numel() * self.patterns.element_size(),
        }


MODEL_FACTORIES: dict[str, Callable[[], Any]] = {
    "classical_hebb": ClassicalHopfield,
    "polynomial_dam_d3": lambda: PolynomialDAM(degree=3),
}
MODEL_FACTORIES

## 3. e：统一测量与结果协议

测量函数只读取目标、终态、轨迹和资源记录，不根据模型名称改变成功判据。U1 的固定点判据是：从干净记忆出发，完成第一轮异步更新后完全不变。

In [ ]:
def e1_exact_recall(final_state: torch.Tensor, target: torch.Tensor) -> bool:
    return bool(torch.equal(final_state.to(torch.int8), target.to(torch.int8)))


def e2_overlap(final_state: torch.Tensor, target: torch.Tensor) -> float:
    return float(
        torch.mean(final_state.to(torch.float64) * target.to(torch.float64)).item()
    )


def e3_top1_memory(final_state: torch.Tensor, memories: torch.Tensor) -> int:
    scores = memories.to(torch.float64) @ final_state.to(torch.float64)
    return int(torch.argmax(scores).item())


def e4_attractor_class(
    final_state: torch.Tensor,
    target_id: int,
    memories: torch.Tensor,
    status: str,
) -> str:
    if status != "fixed":
        return "nonconverged"
    final = final_state.to(torch.int8)
    binary_memories = memories.to(torch.int8)
    matches = torch.all(binary_memories == final.unsqueeze(0), dim=1)
    matched_ids = torch.nonzero(matches, as_tuple=False).flatten().tolist()
    if target_id in matched_ids:
        return "target"
    if matched_ids:
        return "wrong_memory"
    if torch.equal(final, -binary_memories[target_id]):
        return "inverse_target"
    return "spurious_fixed"

## 4. 配对运行器

扫描范围、重复数和停止上限都在运行前固定。异常、达到最大步数和删失不会被静默删除。`run_id + case fields` 相同的记录必须包含全部模型。

In [ ]:
@dataclass(frozen=True)
class BenchmarkConfig:
    N_values: tuple[int, ...]
    P_values: tuple[int, ...]
    corruption_levels: tuple[float, ...]
    pattern_sets: int
    targets_per_set: int
    max_sweeps: int
    base_seed: int
    experiment_id: str
    source_commit: str

    def validate(self) -> None:
        if not self.N_values or not self.P_values or not self.corruption_levels:
            raise ValueError("scan dimensions must not be empty")
        if min(self.N_values) <= 0 or min(self.P_values) <= 0:
            raise ValueError("N and P must be positive")
        if not all(0.0 <= level <= 1.0 for level in self.corruption_levels):
            raise ValueError("corruption levels must lie in [0, 1]")
        if self.pattern_sets <= 0 or self.targets_per_set <= 0 or self.max_sweeps <= 0:
            raise ValueError("replicate counts and max_sweeps must be positive")


def stable_seed(base: int, *coordinates: int) -> int:
    value = int(base) & 0x7FFFFFFF
    for coordinate in coordinates:
        value = (1_103_515_245 * value + 12_345 + int(coordinate)) & 0x7FFFFFFF
    return value


PAIR_KEY = [
    "run_id", "experiment_id", "source_commit", "task_id",
    "pattern_set_id", "target_id", "dataset_id", "encoding",
    "N", "P", "corruption_kind", "corruption_level",
    "data_seed", "cue_seed", "update_seed", "max_sweeps",
    "success_criterion", "retrieval_budget",
    "resource_budget_type", "stopping_rule",
]


def validate_paired_results(
    frame: pd.DataFrame,
    expected_models: Iterable[str] | None = None,
) -> None:
    missing = set(PAIR_KEY).difference(frame.columns)
    if missing or frame.empty:
        raise ValueError(f"invalid result table; missing={sorted(missing)}")
    expected = set(expected_models or sorted(frame["model_id"].unique()))
    for key, group in frame.groupby(PAIR_KEY, dropna=False, sort=False):
        observed = list(group["model_id"])
        if len(observed) != len(set(observed)) or set(observed) != expected:
            raise ValueError(f"unpaired trial {key}: observed={observed}")


def failure_row(
    common: dict[str, Any],
    model_id: str,
    reason: str,
) -> dict[str, Any]:
    return {
        **common,
        "model_id": model_id,
        "model_config": "unavailable",
        "status": "numerical_failure",
        "sweeps": 0,
        "state_updates": 0,
        "one_sweep_unchanged": False,
        "exact_recall": False,
        "overlap": float("nan"),
        "mse": float("nan"),
        "top1_id": float("nan"),
        "attractor_class": "nonconverged",
        "parameter_count": float("nan"),
        "storage_bytes": float("nan"),
        "retrieval_flops": float("nan"),
        "wall_time_ms": float("nan"),
        "right_censored": False,
        "failure_reason": reason,
        "paper_reported": False,
        "energy_trace": [],
        "error_trace": [],
    }


def run_paired_benchmark(
    model_factories: dict[str, Callable[[], Any]],
    config: BenchmarkConfig,
) -> pd.DataFrame:
    config.validate()
    if len(model_factories) < 2:
        raise ValueError("at least two models are required")
    rows: list[dict[str, Any]] = []
    case_index = 0
    for N_index, N in enumerate(config.N_values):
        for P_index, P in enumerate(config.P_values):
            for set_index in range(config.pattern_sets):
                data_seed = stable_seed(config.base_seed, N_index, P_index, set_index)
                memories = a1_make_independent_binary(N, P, data_seed)
                pattern_set_id = f"N{N}-P{P}-set{set_index}-seed{data_seed}"
                fitted: dict[str, Any] = {}
                fit_failures: dict[str, str] = {}
                for registered_id, factory in model_factories.items():
                    try:
                        model = factory()
                        if model.model_id != registered_id:
                            raise ValueError("registry key and model_id differ")
                        fitted[registered_id] = model.fit(memories.patterns)
                    except Exception as error:
                        fit_failures[registered_id] = f"{type(error).__name__}: {error}"
                for target_id in range(min(P, config.targets_per_set)):
                    target = memories.patterns[target_id]
                    for level_index, level in enumerate(config.corruption_levels):
                        cue_seed = stable_seed(
                            config.base_seed, N_index, P_index, set_index,
                            target_id, level_index,
                        )
                        update_seed = stable_seed(cue_seed, 91)
                        cue = c1_make_hamming_cue(target, level, cue_seed)
                        common = {
                            "run_id": f"{config.experiment_id}-{case_index:07d}",
                            "experiment_id": config.experiment_id,
                            "source_commit": config.source_commit,
                            "source_notebook": SOURCE_NOTEBOOK,
                            "task_id": (
                                "U1_clean_fixed_point"
                                if level == 0.0
                                else "U2_noise_recovery"
                            ),
                            "pattern_set_id": pattern_set_id,
                            "target_id": target_id,
                            "dataset_id": memories.dataset_id,
                            "encoding": memories.encoding,
                            "N": N,
                            "P": P,
                            "corruption_kind": "hamming_flip",
                            "corruption_level": float(level),
                            "data_seed": data_seed,
                            "structure_seed": None,
                            "cue_seed": cue_seed,
                            "update_seed": update_seed,
                            "max_sweeps": config.max_sweeps,
                            "success_criterion": (
                                "one_sweep_unchanged"
                                if level == 0.0
                                else "exact_recall"
                            ),
                            "retrieval_budget": "equal_max_sweeps",
                            "resource_budget_type": "native",
                            "stopping_rule": "fixed_or_max_sweeps",
                        }
                        case_index += 1
                        for model_id in model_factories:
                            if model_id in fit_failures:
                                rows.append(failure_row(common, model_id, fit_failures[model_id]))
                                continue
                            model = fitted[model_id]
                            try:
                                started = perf_counter()
                                result = model.retrieve(
                                    cue,
                                    target=target,
                                    update_seed=update_seed,
                                    max_sweeps=config.max_sweeps,
                                )
                                elapsed_ms = (perf_counter() - started) * 1_000.0
                                final_f = result.final_state.to(torch.float64)
                                target_f = target.to(torch.float64)
                                rows.append({
                                    **common,
                                    "model_id": model_id,
                                    **model.resource_summary(),
                                    "status": result.status,
                                    "sweeps": result.sweeps,
                                    "state_updates": result.state_updates,
                                    "one_sweep_unchanged": bool(
                                        result.diagnostics["first_sweep_unchanged"]
                                    ),
                                    "exact_recall": e1_exact_recall(result.final_state, target),
                                    "overlap": e2_overlap(result.final_state, target),
                                    "mse": float((final_f - target_f).pow(2).mean().item()),
                                    "top1_id": e3_top1_memory(result.final_state, memories.patterns),
                                    "attractor_class": e4_attractor_class(
                                        result.final_state, target_id, memories.patterns, result.status
                                    ),
                                    "retrieval_flops": result.retrieval_flops,
                                    "wall_time_ms": elapsed_ms,
                                    "right_censored": False,
                                    "failure_reason": "",
                                    "paper_reported": False,
                                    "energy_trace": result.energy_trace,
                                    "error_trace": result.error_trace,
                                })
                            except Exception as error:
                                reason = f"{type(error).__name__}: {error}"
                                rows.append(failure_row(common, model_id, reason))
    frame = pd.DataFrame(rows)
    validate_paired_results(frame, tuple(model_factories))
    return frame


def capacity_summary(frame: pd.DataFrame, threshold: float = 0.9) -> pd.DataFrame:
    if not 0.0 < threshold < 1.0:
        raise ValueError("threshold must lie in (0, 1)")
    clean = frame[frame["corruption_level"] == 0.0]
    rates = (
        clean.groupby(["model_id", "N", "P"], as_index=False)["one_sweep_unchanged"]
        .mean()
        .rename(columns={"one_sweep_unchanged": "success_rate"})
    )
    rows = []
    for (model_id, N), group in rates.groupby(["model_id", "N"], sort=False):
        ordered = group.sort_values("P")
        passing = ordered[ordered["success_rate"] >= threshold]
        if passing.empty:
            critical = int(ordered["P"].min())
            left_censored, right_censored = True, False
        else:
            critical = int(passing["P"].max())
            left_censored = False
            right_censored = critical == int(ordered["P"].max())
        rows.append({
            "model_id": model_id, "N": int(N), "P_c": critical,
            "success_threshold": threshold,
            "left_censored": left_censored,
            "right_censored": right_censored,
            "capacity_kind": "clean_fixed_point_capacity",
        })
    return pd.DataFrame(rows)

## 5. f：同图比较组件

颜色只编码模型。容量曲线使用实际 `P`；不同增长阶的模型不会被强制放到同一个 `P/N` 横轴。失败记录保留在分母中，容量上下界用删失符号表示。

In [ ]:
MODEL_LABELS = {
    "classical_hebb": "Classical Hopfield",
    "polynomial_dam_d3": "Polynomial DAM (d=3)",
}


def curve_table(frame: pd.DataFrame, x: str, fixed: dict[str, Any]) -> pd.DataFrame:
    subset = frame.copy()
    for column, value in fixed.items():
        subset = subset[subset[column] == value]
    validate_paired_results(subset)
    curve = (
        subset.groupby(["model_id", x])["exact_recall"]
        .agg(["mean", "count"])
        .reset_index()
    )
    curve["se"] = np.sqrt(curve["mean"] * (1 - curve["mean"]) / curve["count"])
    curve["lower"] = np.clip(curve["mean"] - 1.96 * curve["se"], 0, 1)
    curve["upper"] = np.clip(curve["mean"] + 1.96 * curve["se"], 0, 1)
    return curve


def plot_binary_curve(
    curve: pd.DataFrame,
    x: str,
    xlabel: str,
    title: str,
    *,
    log_x: bool = False,
) -> None:
    fig, axis = plt.subplots(figsize=(7.2, 4.4))
    for model_id, group in curve.groupby("model_id", sort=False):
        ordered = group.sort_values(x)
        axis.plot(ordered[x], ordered["mean"], marker="o", label=MODEL_LABELS[model_id])
        axis.fill_between(ordered[x], ordered["lower"], ordered["upper"], alpha=0.18)
    axis.set(xlabel=xlabel, ylabel="Exact recall rate", ylim=(-0.03, 1.03))
    if log_x:
        axis.set_xscale("log", base=2)
    axis.set_title(title)
    axis.legend()
    fig.tight_layout()


def plot_capacity(summary: pd.DataFrame) -> None:
    fig, axis = plt.subplots(figsize=(7.2, 4.4))
    for model_id, group in summary.groupby("model_id", sort=False):
        ordered = group.sort_values("N")
        axis.plot(ordered["N"], ordered["P_c"], marker="o", label=MODEL_LABELS[model_id])
        for marker, column in [("^", "right_censored"), ("v", "left_censored")]:
            censored = ordered[ordered[column]]
            axis.scatter(
                censored["N"], censored["P_c"], marker=marker, s=90,
                facecolors="none", edgecolors="black",
            )
    axis.set(xlabel="State dimension N", ylabel="Empirical critical capacity P_c")
    axis.set_yscale("log", base=2)
    axis.set_title("U3 capacity scaling | one-sweep-unchanged ≥ 90%")
    axis.legend()
    fig.tight_layout()


def plot_dynamics(frame: pd.DataFrame, run_id: str) -> None:
    subset = frame[frame["run_id"] == run_id]
    validate_paired_results(subset)
    fig, axis = plt.subplots(figsize=(7.2, 4.4))
    for row in subset.itertuples(index=False):
        axis.plot(
            range(len(row.error_trace)), row.error_trace, marker="o",
            label=MODEL_LABELS[row.model_id],
        )
    axis.set(
        xlabel="Completed asynchronous sweeps", ylabel="Bit error ratio",
        ylim=(-0.03, 1.03),
    )
    axis.set_title(f"U5 paired dynamics | {run_id}")
    axis.legend()
    fig.tight_layout()


def plot_quality_cost(frame: pd.DataFrame, N: int, P: int, level: float) -> None:
    subset = frame[
        (frame["N"] == N)
        & (frame["P"] == P)
        & (frame["corruption_level"] == level)
        & frame["retrieval_flops"].notna()
    ]
    validate_paired_results(subset)
    grouped = subset.groupby("model_id", as_index=False).agg(
        exact_recall=("exact_recall", "mean"),
        retrieval_flops=("retrieval_flops", "mean"),
    )
    fig, axis = plt.subplots(figsize=(7.2, 4.4))
    for row in grouped.itertuples(index=False):
        axis.scatter(row.retrieval_flops, row.exact_recall, s=80)
        axis.annotate(
            MODEL_LABELS[row.model_id], (row.retrieval_flops, row.exact_recall),
            xytext=(5, 5), textcoords="offset points",
        )
    axis.set(
        xlabel="Approximate retrieval FLOPs", ylabel="Exact recall rate",
        ylim=(-0.03, 1.03),
    )
    axis.set_xscale("log")
    axis.set_title(f"U7 quality-cost | N={N}, P={P}, corruption={level:.2f}")
    fig.tight_layout()

## 6. 公共组件自检

这不是实验结论。它只检查两个模型能接收同一个 memory/cue、返回统一字段，并验证各自记录的能量在数值精度内不增加。

In [ ]:
check_memories = a1_make_independent_binary(N=32, P=4, data_seed=7)
check_target = check_memories.patterns[0]
check_cue = c1_make_hamming_cue(check_target, 0.125, cue_seed=11)
for registered_id, factory in MODEL_FACTORIES.items():
    check_model = factory().fit(check_memories.patterns)
    assert check_model.model_id == registered_id
    check_result = check_model.retrieve(
        check_cue, target=check_target, update_seed=13, max_sweeps=8
    )
    deltas = torch.diff(torch.tensor(check_result.energy_trace))
    assert bool(torch.all(deltas <= 1e-10))
    print(registered_id, check_result.status, check_result.sweeps)
print("component contract: passed")

## 7. Gate 1 有限扫描配置

扫描有明确上限，不自动扩大。它只用于检查基线和同图接口，不足以估计渐近容量指数。当前是 `native / equal_max_sweeps`，还不是严格的同存储或同 FLOPs 比较。

In [ ]:
EXECUTION_ID = (
    datetime.now(timezone.utc).strftime("phase1-%Y%m%dT%H%M%SZ-")
    + uuid4().hex[:8]
)
CONFIG = BenchmarkConfig(
    N_values=(64, 128),
    P_values=(4, 8, 16, 32),
    corruption_levels=(0.0, 0.1, 0.2, 0.3),
    pattern_sets=3,
    targets_per_set=4,
    max_sweeps=20,
    base_seed=20260905,
    experiment_id=EXECUTION_ID,
    source_commit=SOURCE_COMMIT,
)
display(pd.Series(asdict(CONFIG), name="value").to_frame())

## 8. 运行并保存原始记录

每一行是一条目标记忆的一次检索。原始输出保存为 JSON Lines，列表轨迹不会被 CSV 静默改形。失败记录仍保留在比较分母中。

In [ ]:
results = run_paired_benchmark(MODEL_FACTORIES, CONFIG)
validate_paired_results(results, tuple(MODEL_FACTORIES))

artifact_root = Path("/content/hopfield-benchmark-results")
artifact_root.mkdir(parents=True, exist_ok=True)
results.to_json(
    artifact_root / "phase1_raw_results.jsonl",
    orient="records", lines=True,
)
with (artifact_root / "phase1_config.json").open("w", encoding="utf-8") as handle:
    json.dump(asdict(CONFIG), handle, ensure_ascii=False, indent=2)
print(f"saved {len(results)} rows to {artifact_root}")
display(results.drop(columns=["energy_trace", "error_trace"]).head(8))
display(results.groupby(["model_id", "status"]).size().rename("count").to_frame())

## 实验 1：U2 噪声恢复

固定 `N=128, P=16`。横轴是同一批 Hamming 损坏线索，纵轴是 exact recall。

In [ ]:
noise_curve = curve_table(results, "corruption_level", {"N": 128, "P": 16})
plot_binary_curve(
    noise_curve, "corruption_level", "Hamming corruption ratio",
    "U2 noise recovery | N=128, P=16 | native budget",
)
plt.show()

In [ ]:
endpoint = noise_curve[noise_curve["corruption_level"] == 0.3]
lines = [
    f"- `{row['model_id']}`：rho=0.30 时 exact recall={row['mean']:.3f}，n={int(row['count'])}。"
    for _, row in endpoint.iterrows()
]
display(Markdown(
    "**本次运行的描述性结论**\n\n" + "\n".join(lines)
    + "\n\n**证据边界**：这里只是有限规模、native budget 的配对结果，不能据此宣称容量阶数或统计优势。"
))

## 实验 2：U3 有限负载曲线

固定 `N=128` 和 10% Hamming 损坏。横轴使用实际存储数 `P`，不把线性、多项式和指数容量强行归一成同一个 `P/N`。

In [ ]:
load_curve = curve_table(results, "P", {"N": 128, "corruption_level": 0.1})
plot_binary_curve(
    load_curve, "P", "Stored patterns P",
    "U3 finite-load curve | N=128, corruption=0.10 | native budget",
    log_x=True,
)
plt.show()

In [ ]:
display(Markdown(
    "**本次运行的描述性结论**：下表是上图对应的条件均值。\n\n"
    "**证据边界**：曲线可能非单调；不在预设扫描点之间插值，也不把最后一个成功点自动称为理论容量。"
))
display(load_curve.pivot(index="model_id", columns="P", values="mean").round(3))

## 实验 3：U1/U3 干净固定点容量

采用预先固定的 90% one-sweep-unchanged 判据。向上空心点表示容量至少达到扫描上界；向下空心点表示容量低于或等于扫描下界。

In [ ]:
capacities = capacity_summary(results, threshold=0.9)
capacities.to_csv(artifact_root / "phase1_capacity_summary.csv", index=False)
display(capacities)
plot_capacity(capacities)
plt.show()

In [ ]:
lines = []
for row in capacities.itertuples(index=False):
    relation = "≤" if row.left_censored else "≥" if row.right_censored else "="
    lines.append(f"- `{row.model_id}`，N={row.N}：P_c {relation} {row.P_c}。")
display(Markdown(
    "**本次运行的描述性结论**\n\n" + "\n".join(lines)
    + "\n\n**证据边界**：这是离散扫描上的经验阈值。删失点不能用于普通容量拟合，也不能支持线性、多项式或指数容量结论。"
))

## 实验 4：U5 配对动力学

同一个 `run_id` 下叠加公共 bit-error 指标。两种模型的能量定义不同，所以不把能量数值混在一个纵轴。

In [ ]:
dynamics_run_id = results[
    (results["N"] == 128)
    & (results["P"] == 16)
    & (results["corruption_level"] == 0.2)
]["run_id"].iloc[0]
plot_dynamics(results, dynamics_run_id)
plt.show()

In [ ]:
lines = []
for row in results[results["run_id"] == dynamics_run_id].itertuples(index=False):
    final_error = row.error_trace[-1] if row.error_trace else float("nan")
    lines.append(
        f"- `{row.model_id}`：status={row.status}，sweeps={row.sweeps}，final bit error={final_error:.3f}。"
    )
display(Markdown(
    "**本次运行的描述性结论**\n\n" + "\n".join(lines)
    + "\n\n**证据边界**：单条轨迹只解释更新过程，不代表总体召回率。"
))

## 实验 5：U7 质量—计算量

横轴是实现登记的近似操作数，不是硬件实测延迟。它用于暴露 native 比较的资源差异，不等同于严格 matched-compute。

In [ ]:
plot_quality_cost(results, N=128, P=16, level=0.1)
plt.show()

In [ ]:
resource_table = (
    results[
        (results["N"] == 128)
        & (results["P"] == 16)
        & (results["corruption_level"] == 0.1)
    ]
    .groupby("model_id", as_index=False)
    .agg(
        exact_recall=("exact_recall", "mean"),
        retrieval_flops=("retrieval_flops", "mean"),
        storage_bytes=("storage_bytes", "mean"),
    )
)
display(Markdown(
    "**本次运行的描述性结论**：同图中的质量差异必须和下面的资源量一起读。\n\n"
    "**证据边界**：当前不做硬件速度结论；向量化程度会改变 wall-clock time。"
))
display(resource_table.round(3))

## 9. 下一道实验门

先在 Colab 验收 Classical Hopfield 的干净固定点、异步能量不增与有限规模容量，再验收 Polynomial DAM 的坐标能量不增和 degree=2 退化关系。两条候选基线通过以后，才加入 matched-storage、matched-compute 与更多模型。

能画在同一张图上只说明任务已经配对，不自动说明预算公平，也不自动支持论文级容量结论。